# Qwen3-0.6B router training
Train on the measured Zen profiles, select by validation loss, and leave the held-out test sealed.

In [ ]:
!nvidia-smi
!git clone https://github.com/shahidhustles/agent-route.git /content/agent-route
%cd /content/agent-route
!python -m pip install -q -r training/requirements.txt
!python training/generate_data.py
!python -m unittest discover -s training -p 'test_data.py'

Run a short GPU smoke test before spending time on the full pipeline.

In [ ]:
!python training/train.py --max-steps 3 --limit 32 --accumulation 1 --output artifacts/smoke

The full pipeline trains, restores the checkpoint with the lowest validation loss, and runs validation only.

In [ ]:
!python -u training/run_pipeline.py

In [ ]:
import json
from pathlib import Path
status = json.loads(Path('artifacts/qwen3-router/pipeline-status.json').read_text())
assert status['stage'] == 'validated', status
status

Download the frozen training and validation evidence. Do not evaluate the held-out test during tuning.

In [ ]:
import shutil
from google.colab import files
artifact = shutil.make_archive('/content/qwen3-router-artifacts', 'zip', '/content/agent-route/artifacts/qwen3-router')
files.download(artifact)